In [44]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from pyvis.network import Network

from dotenv import load_dotenv
import os
import asyncio
from langchain.chat_models import init_chat_model
os.environ["DEEPSEEK_API_KEY"] = "sk-ae6d423f13b7403fa8ba16ddb4acc297"

In [2]:
llm=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

In [3]:
# Load the .env file
load_dotenv()
# Get API key from environment variable
api_key = os.getenv("DEEPSEEK_API_KEY")

In [4]:
# llm = ChatOpenAI(temperature=0, model_name="gpt-4o")
llm=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

graph_transformer = LLMGraphTransformer(llm=llm)

In [6]:
# 1. 读取 md 并按段落切分
def load_md_and_split(md_path):
    """
    输入：md 文件路径
    输出：段落列表（每段是一段连续文本）
    """
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip() and p[0] != '#']
    return paragraphs

In [14]:
# Extract graph data from input text
# 一次性输入整本书。最好不超过80000字。会整个送入llm
# text-->graph
async def extract_graph_data(text):
    """
    Asynchronously extracts graph data from input text using a graph transformer.

    Args:
        text (str): Input text to be processed into graph format.

    Returns:
        list: A list of GraphDocument objects containing nodes and relationships.
    """
    documents = [Document(page_content=text)]
    graph_documents = await graph_transformer.aconvert_to_graph_documents(documents)
    return graph_documents

In [39]:
# chunks-->subgraphs
# chunks: list(str)
async def extract_graph_data_from_chunks(chunks):
    """
    从分段文本列表中提取知识图谱，每段独立调用 LLM，最后合并结果。

    Args:
        chunks (List[str]): 分段后的文本列表，每段是一个字符串。

    Returns:
        List[GraphDocument]: 合并后的 GraphDocument 列表（通常为一个或多个）
    """
    all_graph_documents = []
    
    for i, chunk in enumerate(chunks):
        if not chunk.strip():  # 跳过空段
            continue
        print(f"Processing chunk {i+1}/{len(chunks)} (length: {len(chunk)} chars)...")
        
        # 封装为 LangChain Document
        documents = [Document(page_content=chunk)]
        
        # 调用 LLM 提取图谱（单段）
        try:
            graph_docs = await graph_transformer.aconvert_to_graph_documents(documents)
            all_graph_documents.extend(graph_docs)
        except Exception as e:
            print(f"⚠️ Chunk {i+1} failed: {str(e)}")
            continue  # 跳过失败段，继续处理其他
    
    print(f"✅ Total extracted: {len(all_graph_documents)} GraphDocument(s)")
    return all_graph_documents

In [48]:
# subgraphs --> graph
def merge_kg_to_dict(graph_docs):
    """
    将 List[GraphDocument] 合并为一个去重的 {nodes: [...], relationships: [...]} 字典
    适合直接 json.dump
    """
    node_dict = {}  # key: node.id
    rel_set = set()  # 存储 (source_id, target_id, rel_type) 用于去重
    rel_list = []

    for doc in graph_docs:
        # 收集节点
        for node in doc.nodes:
            node_dict[node.id] = {
                "id": node.id,
                "type": node.type,
                "properties": dict(node.properties) if node.properties else {}
            }
        
        # 收集关系（去重）
        for rel in doc.relationships:
            key = (rel.source.id, rel.target.id, rel.type)
            if key not in rel_set:
                rel_set.add(key)
                rel_list.append({
                    "source": rel.source.id,
                    "target": rel.target.id,
                    "type": rel.type,
                    "properties": dict(rel.properties) if rel.properties else {}
                })
    
    return {
        "nodes": list(node_dict.values()),
        "relationships": rel_list
    }

In [22]:
import json

def graph_documents_to_dict(graph_docs):
    """
    将 List[GraphDocument] 转换为可序列化的字典列表（忽略 source Document）
    """
    result = []
    for doc in graph_docs:
        nodes_list = [
            {
                "id": node.id,
                "type": node.type,
                "properties": node.properties or {}
            }
            for node in doc.nodes
        ]
        
        relationships_list = [
            {
                "source": rel.source.id,
                "target": rel.target.id,
                "type": rel.type,
                "properties": rel.properties or {}
            }
            for rel in doc.relationships
        ]
        
        result.append({
            "nodes": nodes_list,
            "relationships": relationships_list
        })
    
    return result

In [31]:
md_path='./data/cleaned_湘教版地理必修一Chap2.md'
text=load_md_and_split(md_path)

In [32]:
# text="\n".join(text)
# kg = await extract_graph_data(text)
kg = await extract_graph_data_from_chunks(text)

Processing chunk 1/147 (length: 171 chars)...
Processing chunk 2/147 (length: 167 chars)...
Processing chunk 3/147 (length: 4 chars)...
Processing chunk 4/147 (length: 17 chars)...
Processing chunk 5/147 (length: 22 chars)...
Processing chunk 6/147 (length: 25 chars)...
Processing chunk 7/147 (length: 56 chars)...
Processing chunk 8/147 (length: 59 chars)...
Processing chunk 9/147 (length: 77 chars)...
Processing chunk 10/147 (length: 39 chars)...
Processing chunk 11/147 (length: 77 chars)...
Processing chunk 12/147 (length: 23 chars)...
Processing chunk 13/147 (length: 138 chars)...
Processing chunk 14/147 (length: 11 chars)...
Processing chunk 15/147 (length: 12 chars)...
Processing chunk 16/147 (length: 104 chars)...
Processing chunk 17/147 (length: 26 chars)...
Processing chunk 18/147 (length: 58 chars)...
Processing chunk 19/147 (length: 48 chars)...
Processing chunk 20/147 (length: 37 chars)...
Processing chunk 21/147 (length: 101 chars)...
Processing chunk 22/147 (length: 9 char

In [42]:
kg

[GraphDocument(nodes=[Node(id='地球', type='天体', properties={}), Node(id='地貌', type='概念', properties={}), Node(id='大自然', type='自然力量', properties={}), Node(id='流水', type='自然现象', properties={}), Node(id='山洪', type='自然现象', properties={}), Node(id='波浪', type='自然现象', properties={}), Node(id='冰川', type='自然现象', properties={}), Node(id='锦绣山河', type='概念', properties={}), Node(id='大千世界', type='概念', properties={})], relationships=[Relationship(source=Node(id='地貌', type='概念', properties={}), target=Node(id='地球', type='天体', properties={}), type='属于', properties={}), Relationship(source=Node(id='地貌', type='概念', properties={}), target=Node(id='地球演化', type='概念', properties={}), type='结果', properties={}), Relationship(source=Node(id='大自然', type='自然力量', properties={}), target=Node(id='流水', type='自然现象', properties={}), type='使用', properties={}), Relationship(source=Node(id='大自然', type='自然力量', properties={}), target=Node(id='山洪', type='自然现象', properties={}), type='使用', properties={}), Relationship(source=No

In [50]:
kg=merge_kg_to_dict(kg)

In [52]:
kg

{'nodes': [{'id': '地球', 'type': '天体', 'properties': {}},
  {'id': '地貌', 'type': '概念', 'properties': {}},
  {'id': '大自然', 'type': '自然力量', 'properties': {}},
  {'id': '流水', 'type': '自然现象', 'properties': {}},
  {'id': '山洪', 'type': '自然现象', 'properties': {}},
  {'id': '波浪', 'type': '自然现象', 'properties': {}},
  {'id': '冰川', 'type': '地理特征', 'properties': {}},
  {'id': '锦绣山河', 'type': '概念', 'properties': {}},
  {'id': '大千世界', 'type': '概念', 'properties': {}},
  {'id': '青居镇', 'type': 'Town', 'properties': {}},
  {'id': '嘉陵江', 'type': 'River', 'properties': {}},
  {'id': '上码头', 'type': 'Location', 'properties': {}},
  {'id': '下码头', 'type': 'Location', 'properties': {}},
  {'id': '曲水', 'type': 'Location', 'properties': {}},
  {'id': '青居人家', 'type': 'Person', 'properties': {}},
  {'id': '纤夫', 'type': 'Person', 'properties': {}},
  {'id': '河流', 'type': 'Geographical_feature', 'properties': {}},
  {'id': '图2-1', 'type': 'Image', 'properties': {}},
  {'id': '地表形态', 'type': 'Landform', 'properties': {

In [51]:
# 假设 kg 是你已经获取到的 GraphDocument 列表
kg_dict = graph_documents_to_dict(kg)

# 保存为 JSON 文件
with open("bychunks_graph_c2.json", "w", encoding="utf-8") as f:
    json.dump(kg_dict, f, ensure_ascii=False, indent=2)

print("知识图谱已保存为 knowledge_graph.json")

AttributeError: 'str' object has no attribute 'nodes'

In [53]:
with open("merged_kg.json", "w", encoding="utf-8") as f:
    json.dump(kg, f, ensure_ascii=False, indent=2)

print("✅ 合并完成，已保存到 merged_kg.json")

✅ 合并完成，已保存到 merged_kg.json


In [56]:
import json
from neo4j import GraphDatabase

# === 配置 Neo4j 连接 ===
uri = "bolt://localhost:7689"        # 改成你的 Neo4j 地址
username = "neo4j"                   # 默认用户名
password = "12345678"           # 修改为你的密码

driver = GraphDatabase.driver(uri, auth=(username, password))

def create_nodes(tx, nodes):
    for node in nodes:
        id_ = node["id"]
        node_type = node["type"].replace(" ", "_")  # Neo4j 标签不能有空格
        props = node.get("properties", {})
        
        # 转义属性名（避免关键字冲突）
        prop_assign = ", ".join([f"n.{k} = ${k}" for k in props.keys()])
        query = f"""
        MERGE (n:`{node_type}` {{id: $id}})
        SET n.id = $id
        """
        if prop_assign:
            query += f" SET {prop_assign}"
        params = {"id": id_, **props}
        tx.run(query, params)

def create_relationships(tx, relationships):
    for rel in relationships:
        source = rel["source"]
        target = rel["target"]
        rel_type = rel["type"].replace(" ", "_")  # 关系类型也不能有空格
        props = rel.get("properties", {})
        
        prop_assign = ", ".join([f"r.{k} = ${k}" for k in props.keys()])
        query = f"""
        MATCH (a {{id: $source}}), (b {{id: $target}})
        MERGE (a)-[r:`{rel_type}`]->(b)
        """
        if prop_assign:
            query += f" SET {prop_assign}"
        params = {"source": source, "target": target, **props}
        tx.run(query, params)

def import_json_to_neo4j(json_file_path):
    """主函数：读取 JSON 并导入 Neo4j（支持顶层为 list 或 dict）"""
    with open(json_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # 如果顶层是 list，则遍历每个元素；如果是 dict，则包装成 list
    if isinstance(data, list):
        graph_list = data
    else:
        graph_list = [data]  # 单个图
    
    all_nodes = []
    all_rels = []
    
    for graph in graph_list:
        all_nodes.extend(graph.get("nodes", []))
        all_rels.extend(graph.get("relationships", []))
    
    # 去重（可选，避免重复插入）
    node_dict = {node["id"]: node for node in all_nodes}
    rel_set = set()
    unique_rels = []
    for rel in all_rels:
        key = (rel["source"], rel["target"], rel["type"])
        if key not in rel_set:
            rel_set.add(key)
            unique_rels.append(rel)
    
    with driver.session() as session:
        session.write_transaction(create_nodes, list(node_dict.values()))
        session.write_transaction(create_relationships, unique_rels)
    
    print("✅ 数据已成功导入 Neo4j！")

# === 执行 ===
if __name__ == "__main__":
    import_json_to_neo4j("new_graph_c2.json")

C:\Users\lenovo\AppData\Local\Temp\ipykernel_22776\3041615337.py:74: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_nodes, list(node_dict.values()))
C:\Users\lenovo\AppData\Local\Temp\ipykernel_22776\3041615337.py:75: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_relationships, unique_rels)


✅ 数据已成功导入 Neo4j！
